# model_experiment_LightGBM

Global gradient-boosting model over all 3331 series. Direct multi-horizon: every lag feature is >= 39 weeks back, so one `predict()` covers the whole 39-week test set with no recursion.

MLflow experiment: **LightGBM_Training** — runs: Cleaning, CV_*, Feature_Importance, Holiday_Blend_*, Final.

Round 2 additions: holiday-aligned lag features (`hol_lag_1y`, `hol_relpos`), naive blend on holiday weeks, ablation submissions (no-shift / shift / shift+blend).

In [ ]:
# Kaggle bootstrap — does nothing when running locally.
# On Kaggle: attach the competition data (Add Input) and the three
# MLFLOW_TRACKING_* secrets (Add-ons -> Secrets) before Run All.
import os
ON_KAGGLE = os.path.exists("/kaggle")
if ON_KAGGLE:
    os.system("pip install -q mlflow lightgbm scikit-learn")
    if not os.path.exists("walmart-sales-forecasting"):
        os.system("git clone https://github.com/ekatsirekidze/walmart-sales-forecasting.git")
    import sys; sys.path.insert(0, "/kaggle/working/walmart-sales-forecasting")
    import glob, shutil, zipfile
    src = glob.glob("/kaggle/input/*walmart*") + glob.glob("/kaggle/input/*/*walmart*")
    assert src, "competition data not attached"
    os.makedirs("data", exist_ok=True)
    for f in glob.glob(src[0] + "/*"):
        (zipfile.ZipFile(f).extractall("data") if f.endswith(".zip") else shutil.copy(f, "data"))
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ["MLFLOW_TRACKING_URI"] = s.get_secret("MLFLOW_TRACKING_URI")
    os.environ["MLFLOW_TRACKING_USERNAME"] = s.get_secret("MLFLOW_TRACKING_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = s.get_secret("MLFLOW_TRACKING_PASSWORD")

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))  # repo root on path (local runs)

import numpy as np
import pandas as pd
import mlflow

from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline

from src.data import load_raw, make_submission
from src.metrics import wmae_report
from src.validation import FOLDS, split_fold
from src.mlflow_utils import setup_mlflow, REGISTRY_MODEL_NAME
from src.preprocessing import WalmartPreprocessor, make_xyw
from src.postprocess import apply_christmas_shift, blend_holiday_naive

train, test, features, stores = load_raw("data" if ON_KAGGLE else None)
print(train.shape, test.shape)

setup_mlflow("LightGBM_Training")

## Run 1 — Cleaning (documenting the data decisions)

In [ ]:
with mlflow.start_run(run_name="LightGBM_Cleaning"):
    mlflow.log_params({
        "negative_sales": "kept (returns are real; metric is L1 on raw dollars)",
        "markdown_na": "fill 0 + <col>_present flag (NA = no promo running)",
        "cpi_unemployment_na": "forward-fill per store (covers test tail)",
        "target_transform": "none (L1 objective on raw sales matches WMAE)",
        "holiday_features": "aligned-by-holiday lag (hol_lag_1y) + relative position",
        "n_rows": len(train),
        "n_series": train.groupby(["Store", "Dept"]).ngroups,
    })

## Run 2 — Cross-validation on the shared folds

`sample_weight = 1 + 4*IsHoliday` so training optimizes what Kaggle measures.

In [ ]:
def eval_fold(params, fold, holiday_lags=False):
    tr, va = split_fold(train, fold)
    Xtr, ytr, wtr = make_xyw(tr)
    Xva, yva, _ = make_xyw(va)
    prep = WalmartPreprocessor(features_df=features, stores_df=stores,
                               holiday_lags=holiday_lags)
    Xtr_t = prep.fit(Xtr, ytr).transform(Xtr)
    Xva_t = prep.transform(Xva)
    model = LGBMRegressor(**params)
    model.fit(Xtr_t, ytr, sample_weight=wtr)
    rep = wmae_report(yva, model.predict(Xva_t), va["IsHoliday"])
    return rep, model, Xtr_t, Xva_t

In [ ]:
# round 2: config 0 won round 1; 255 leaves overfit holidays -> stay <=127,
# explore more regularization instead
PARAM_GRID = [
    dict(objective="l1", n_estimators=800, learning_rate=0.05, num_leaves=127,
         min_child_samples=30, colsample_bytree=0.8, random_state=42, verbose=-1),
    dict(objective="l1", n_estimators=1200, learning_rate=0.04, num_leaves=127,
         min_child_samples=20, colsample_bytree=0.7, random_state=42, verbose=-1),
    dict(objective="l1", n_estimators=1200, learning_rate=0.04, num_leaves=127,
         min_child_samples=30, colsample_bytree=0.7, subsample=0.9, subsample_freq=1,
         random_state=42, verbose=-1),
]

results = []
for i, params in enumerate(PARAM_GRID):
    with mlflow.start_run(run_name=f"LightGBM_CV_r2_{i}"):
        mlflow.log_params({**params, "fold": 1, "round": 2})
        rep, model, _, _ = eval_fold(params, fold=1)
        mlflow.log_metrics(rep)
    results.append((i, rep["wmae"]))
    print(i, rep)
best_i = min(results, key=lambda r: r[1])[0]
print("best config:", best_i)

In [ ]:
# sanity-check the winner on Fold 2 (regular weeks, no big holidays)
with mlflow.start_run(run_name="LightGBM_CV_r2_best_fold2"):
    mlflow.log_params({**PARAM_GRID[best_i], "fold": 2})
    rep2, _, _, _ = eval_fold(PARAM_GRID[best_i], fold=2)
    mlflow.log_metrics(rep2)
print(rep2)

## Run 2b — Holiday-aligned lag features: negative ablation

Idea: the big holidays drift across week numbers, so align lags by holiday instead of a fixed 52-week offset. Result on Fold 1: **it hurts** (holiday MAE 2355 → 2464) — one aligned year is too noisy and the model over-trusts it. Logged as a documented negative result; feature stays off.

In [ ]:
with mlflow.start_run(run_name="LightGBM_HolidayLags_Ablation"):
    rep_hl, _, _, _ = eval_fold(PARAM_GRID[best_i], fold=1, holiday_lags=True)
    mlflow.log_params({**PARAM_GRID[best_i], "fold": 1, "holiday_lags": True})
    mlflow.log_metrics(rep_hl)
print("with holiday_lags:", rep_hl)
print("without (CV winner above): kept — holiday_lags stays OFF")

## Run 3 — Feature importance

In [ ]:
rep, model, Xtr_t, Xva_t = eval_fold(PARAM_GRID[best_i], fold=1)
imp = pd.Series(model.feature_importances_, index=Xtr_t.columns).sort_values(ascending=False)
with mlflow.start_run(run_name="LightGBM_Feature_Importance"):
    mlflow.log_dict(imp.to_dict(), "feature_importance.json")
    mlflow.log_metrics(rep)
print(imp.head(20))

## Run 4 — Holiday blend weight search (Fold 1), Christmas EXCLUDED

Round-1 finding: naive lag-52 beat the model on holiday weeks (MAE 2300 vs 2355) → blend model with naive on holiday weeks.

Round-2 LB finding: blending the **Christmas** week backfires — naive copies last year's post-Christmas week, whose pre-Christmas-shopping-day count differs across years (0/1/3 days), so the blend undoes the Christmas-shift fix (2604 public with Christmas blended vs 2507 without any blend). Confirmed on Fold 1 too: excluding Christmas improves holiday MAE 2159 → 2102. So the blend applies to Super Bowl / Labor Day / Thanksgiving weeks only.

In [ ]:
from src.preprocessing import BLEND_HOLIDAY_WEEKS   # SB + LD + TG, no Christmas

tr, va = split_fold(train, 1)
_, yva, _ = make_xyw(va)
va_pred = va[["Store", "Dept", "Date"]].copy()
va_pred["pred"] = model.predict(Xva_t)   # best model from the importance cell

blend_scores = {}
for w in (0.0, 0.5, 0.6, 0.75, 0.9):
    blended = blend_holiday_naive(va_pred, tr, weight=w, holiday_dates=BLEND_HOLIDAY_WEEKS)
    rep = wmae_report(yva, blended["pred"], va["IsHoliday"])
    blend_scores[w] = rep["wmae"]
    with mlflow.start_run(run_name=f"LightGBM_Blend_noXmas_w{w}"):
        mlflow.log_params({**PARAM_GRID[best_i], "blend_weight": w,
                           "blend_weeks": "SB+LD+TG (no Christmas)", "fold": 1})
        mlflow.log_metrics(rep)
    print(f"w={w}: {rep}")

BLEND_W = min(blend_scores, key=blend_scores.get)
print("best blend weight:", BLEND_W)

## Run 5 — Final: fit on ALL train, register, submissions

Config selection note: Fold-1 preferred config 1 (1901 vs 1925) but the leaderboard preferred config 0 (2567 vs 2592 private) — a ~25-point gap in both directions, i.e. single-fold selection noise. We submit BOTH configs with the fixed blend and let the leaderboard decide; config 0 is registered as primary (better test-period evidence).

In [ ]:
X, y, w = make_xyw(train)
CFG0 = PARAM_GRID[0]

pipe = Pipeline([
    ("prep", WalmartPreprocessor(features_df=features, stores_df=stores)),
    ("model", LGBMRegressor(**CFG0)),
])
pipe.fit(X, y, model__sample_weight=w)

with mlflow.start_run(run_name="LightGBM_Final"):
    mlflow.log_params({**CFG0, "blend_weight": BLEND_W,
                       "blend_weeks": "SB+LD+TG (no Christmas)",
                       "selection": "cfg0 by LB evidence; fold-1 diff was noise"})
    mlflow.log_metric("fold1_wmae", dict(results)[0])
    # if LightGBM ends up the overall best architecture, add:
    #   registered_model_name=REGISTRY_MODEL_NAME
    mlflow.sklearn.log_model(pipe, name="model",
                             serialization_format="cloudpickle",
                             registered_model_name="walmart-lightgbm")

In [ ]:
# raw test in -> submissions out: both configs, shift + no-Christmas blend
def make_final_sub(params, fname):
    p = Pipeline([
        ("prep", WalmartPreprocessor(features_df=features, stores_df=stores)),
        ("model", LGBMRegressor(**params)),
    ])
    p.fit(X, y, model__sample_weight=w)
    sub = test[["Store", "Dept", "Date"]].copy()
    sub["pred"] = p.predict(test[["Store", "Dept", "Date"]])
    sub = apply_christmas_shift(sub, pred_col="pred")
    sub = blend_holiday_naive(sub, train, weight=BLEND_W,
                              holiday_dates=BLEND_HOLIDAY_WEEKS)
    make_submission(sub, "pred", fname)
    print("wrote", fname)

make_final_sub(PARAM_GRID[0], "submission_lgbm_cfg0_shift_blendnx.csv")
make_final_sub(PARAM_GRID[1], "submission_lgbm_cfg1_shift_blendnx.csv")